In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install fastdtw gensim nltk kmedoids tensorflow

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

In [ ]:
from __future__ import annotations
import gc
import pickle
import random
import string
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Sequence, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Bidirectional,
    Conv1D,
    Dense,
    Dropout,
    Layer,
    LSTM,
    MaxPooling1D,
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from fastdtw import fastdtw
from gensim.models import KeyedVectors, Word2Vec
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.ensemble import IsolationForest
import kmedoids as km
warnings.filterwarnings("ignore")

In [ ]:
"""Manages project and output directories for the pipeline.

    Attributes:
        project_root: Root directory containing raw data and assets (Torah, imposters).
        results_root: Output directory where progress and results are saved.
        hebrew_fasttext_path: Path to the Hebrew FastText embedding vectors.
        progress_file: JSON file tracking execution progress.
    """
@dataclass
class PipelinePaths:
    project_root: Path
    results_root: Path

    torah_dir: Path = field(init=False)
    imposters_dir: Path = field(init=False)
    imposters1_dir: Path = field(init=False)
    hebrew_fasttext_path: Path = field(init=False)
    progress_file: Path = field(init=False)

    def __post_init__(self):
        self.torah_dir = self.project_root / "torah_chapters"
        self.imposters_dir = self.project_root / "imposters"
        self.imposters1_dir = self.project_root / "imposters1"
        self.hebrew_fasttext_path = self.project_root / "cc.he.300.vec.gz"
        self.results_root.mkdir(parents=True, exist_ok=True)
        self.progress_file = self.results_root / "progress.json"

"""Holds hyperparameters for text processing, model architecture, and evaluation.

    Attributes:
        chunk_size & batch_factor: Define text splitting and the input sequence length.
        kernel_sizes & num_filters: CNN architecture configurations for feature extraction.
        lstm_units & dense_units: Hidden layer dimensions for sequential and dense layers.
        isolation_threshold: Anomaly detection cutoff for analysis.
        sequence_length: Property calculating the final input length (chunk_size // batch_factor).
    """
@dataclass
class ExperimentConfig:
    embedding_dimension: int = 300
    chunk_size: int = 100
    batch_factor: int = 2

    kernel_sizes: Tuple[int, ...] = (3, 6, 12)
    num_filters: int = 500
    pool_size: int = 1
    dense_units: int = 2048
    num_classes: int = 2
    learning_rate: float = 0.0001

    epochs: int = 10
    lstm_units: int = 1200

    test_size: float = 0.25
    dropout_rate: float = 0.50
    isolation_threshold: float = 0.00005 / 2
    summary_percentile: int = 90
    embedding_algorithm: str = "hebrew_fasttext_vec"

    @property
    def sequence_length(self) -> int:
        return self.chunk_size // self.batch_factor

"""Tracks dynamic state, metrics, and clustering results across iterations.

    Attributes:
        iteration_number: The current execution round.
        database: Key-value store holding processed text tokens.
        forest_report & summary_report: Dynamic evaluation metrics and anomaly scores.
        latest_distance_matrix: The most recently computed pairwise distance matrix.
        latest_kmedoids_labels: Cluster assignments from the latest K-Medoids run.
    """
@dataclass
class PipelineState:

    iteration_number: int = 1
    database: Dict[str, List[str]] = field(default_factory=dict)

    forest_report: Dict[object, Sequence[float]] = field(default_factory=dict)

    summary_report: Dict[object, Sequence[float]] = field(default_factory=dict)

    distance_label_report: Dict[object, Sequence[int]] = field(default_factory=dict)

    latest_distance_matrix: np.ndarray | None = None

    latest_kmedoids_labels: np.ndarray | None = None

In [ ]:
"""Adapter for K-Medoids clustering using the FasterPAM algorithm.

    Attributes:
        n_clusters: The target number of clusters to form.
        metric: Distance metric strategy; strictly expects 'precomputed'.
        init: Medoid initialization method (e.g., 'heuristic').
        labels_: Array containing cluster assignments for each data point after fitting.
        medoid_indices_: Array containing indices of the chosen cluster medoids.
    """
class KMedoidsAdapter:
    def __init__(
        self,
        n_clusters: int = 8,
        metric: str = "precomputed",
        init: str = "heuristic",
        max_iter: int = 300,
        random_state: int | None = None,
    ):

        self.n_clusters = n_clusters
        self.metric = metric
        self.init = init
        self.max_iter = max_iter
        self.random_state = random_state
        self.labels_ = None
        self.medoid_indices_ = None

    def fit(self, distance_matrix: np.ndarray):
        if self.metric != "precomputed":
            raise ValueError("This pipeline expects a precomputed distance matrix.")

        result = km.fasterpam(
            diss=np.asarray(distance_matrix),
            medoids=self.n_clusters,
            max_iter=self.max_iter,
            init=self.init,
            random_state=self.random_state,
        )

        self.labels_ = np.asarray(result.labels)
        self.medoid_indices_ = np.asarray(result.medoids)
        return self

"""Custom Attention layer for 2D or 3D tensors, supporting sequential output.

    Attributes:
        return_sequences: If True, returns weighted sequences; if False, averages them over time.
        weight_matrix: Trainable weight matrix mapping input features to attention logits.
        bias: Trainable bias vector for the attention computation.
    """
class Attention(Layer):

    def __init__(self, return_sequences: bool = True, **kwargs):

        super().__init__(**kwargs)
        self.return_sequences = return_sequences

    def build(self, input_shape):

        self.weight_matrix = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.bias = self.add_weight(
            name="att_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs):

        if len(inputs.shape) == 2:
            attention_logits = tf.math.tanh(tf.matmul(inputs, self.weight_matrix) + self.bias)
            attention_weights = tf.nn.softmax(attention_logits, axis=1)
            return inputs * attention_weights

        if len(inputs.shape) == 3:
            attention_logits = tf.math.tanh(tf.matmul(inputs, self.weight_matrix) + self.bias)
            attention_weights = tf.nn.softmax(attention_logits, axis=1)
            weighted_output = inputs * attention_weights

            if self.return_sequences:
                return weighted_output

            return tf.reduce_sum(weighted_output, axis=1)

        raise ValueError(f"Unsupported tensor rank: {len(inputs.shape)}")

In [ ]:
"""Repository class for managing data loading and directory traversal for the corpus.

    Attributes:
        paths: PipelinePaths instance containing the source and results directories.
        state: PipelineState instance where the loaded text data is stored.
        _read_all_texts: Static helper that reads and normalizes all text files in a directory.
    """
class CorpusRepository:
    def __init__(self, paths: PipelinePaths, state: PipelineState):
        self.paths = paths
        self.state = state

    def load_torah_corpus(self):
        # טעינת ספרי התורה (בראשית, שמות וכו')
        self.state.database["torah"] = self._read_all_texts(self.paths.torah_dir)

    def load_imposter_group_by_name(self, author_name: str, group_name: str):
        if group_name == "imposters":
            source_dir = self.paths.imposters_dir / author_name
        else:
            source_dir = self.paths.imposters1_dir / author_name
        self.state.database[author_name] = self._read_all_texts(source_dir)

    def list_imposter_names(self, group_name: str) -> List[str]:
        source_dir = self.paths.imposters_dir if group_name == "imposters" else self.paths.imposters1_dir
        return sorted([item.name for item in source_dir.iterdir() if item.is_dir()])

    def list_torah_file_stems(self) -> List[str]:
        return sorted([file.stem for file in self.paths.torah_dir.glob("*.txt")])

    @staticmethod
    def _read_all_texts(folder: Path) -> List[str]:
        texts = []
        for file_path in sorted(folder.glob("*.txt")):
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                texts.append(f.read().replace("\n", " "))
        return texts

In [ ]:
import re
import string

"""Handles text cleaning, diacritic removal, and tokenization for Hebrew corpora.

    Attributes:
        punctuation_table: Translation table for stripping standard and Hebrew punctuation.
        clean_text: Cleans a string by removing non-Hebrew characters and diacritics.
        preprocess_collection: Processes a collection of texts into filtered Hebrew token lists.
    """
class TextPreprocessor:

    def __init__(self):

        self.punctuation_table = str.maketrans("", "", string.punctuation + '״' + '׳')

    def remove_hebrew_diacritics(self, text: str) -> str:

        return re.sub(r'[\u0591-\u05C7]', '', text)

    def remove_english_letters(self, text: str) -> str:

        return re.sub(r'[A-Za-z]', '', text)

    def clean_text(self, text: str) -> str:

        text = text.replace("\n", " ")

        text = self.remove_hebrew_diacritics(text)

        text = self.remove_english_letters(text)

        text = text.translate(self.punctuation_table)

        text = re.sub(r'[^א-ת\s]', ' ', text)

        text = re.sub(r'\s+', ' ', text).strip()

        return text

    def preprocess_collection(self, texts):
        processed_texts = []

        for text in texts:
            text = self.clean_text(text)

            tokens = text.split()

            tokens = [token for token in tokens if re.fullmatch(r'[א-ת]+', token)]

            processed_texts.append(tokens)

        return processed_texts

"""Loads pretrained vectors and trains a Word2Vec model intersected with FastText knowledge.

    Attributes:
        paths: PipelinePaths instance indicating where embedding assets are stored.
        pretrained_vectors: Cached Hebrew FastText vectors loaded into memory.
        train_word2vec: Builds vocabulary and initializes local weights using pretrained embeddings.
    """
class EmbeddingTrainer:

    def __init__(self, paths: PipelinePaths):
        self.paths = paths
        self.pretrained_vectors = None

    def _ensure_vectors_loaded(self):
        if self.pretrained_vectors is None:
            print("--- [RAM] Loading Hebrew FastText into memory (Once)... ---")
            self.pretrained_vectors = KeyedVectors.load_word2vec_format(
                str(self.paths.hebrew_fasttext_path),
                binary=False
            )
            print(f"--- [RAM] Successfully loaded {len(self.pretrained_vectors.index_to_key)} vectors ---")

    def train_word2vec(self, imp_a, imp_b, torah_tokens, vector_size, algorithm) -> Word2Vec:
        self._ensure_vectors_loaded()

        full_training_corpus = imp_a + imp_b + torah_tokens

        model = Word2Vec(vector_size=vector_size, min_count=1)
        model.build_vocab(full_training_corpus)

        model.wv.vectors_lockf = np.ones(len(model.wv), dtype=np.float32)

        print(f"--- [Logic] Intersecting weights from RAM for {len(model.wv.index_to_key)} local words ---")

        count = 0
        for i, word in enumerate(model.wv.index_to_key):
            if word in self.pretrained_vectors:
                model.wv.vectors[i] = self.pretrained_vectors[word]
                count += 1

        print(f"--- [Logic] Success: {count} words initialized from FastText knowledge ---")

        model.train(
            full_training_corpus,
            total_examples=model.corpus_count,
            epochs=1
        )

        return model

    def _load_google_news_vectors(self) -> KeyedVectors:

        if self.paths.pretrained_google_news_path.exists():
            return KeyedVectors.load_word2vec_format(
                str(self.paths.pretrained_google_news_path),
                binary=True,
            )

        raise FileNotFoundError("GoogleNews pretrained vectors were not found.")

"""Transforms tokenized texts into structured, fixed-length 3D embedding blocks for neural networks.

    Attributes:
        tokens_to_embedding_blocks: Static method converting a flat token list into shaped vector blocks.
        collection_to_embedding_blocks: Efficiently vectorizes a whole text collection using bulk indexing.
    """
class SequenceEmbedder:

    @staticmethod
    def tokens_to_embedding_blocks(
        tokens: Sequence[str],
        embedding_model: Word2Vec,
        sequence_length: int,
    ) -> np.ndarray:

        word_vectors = embedding_model.wv
        word_to_index = word_vectors.key_to_index
        vector_size = word_vectors.vector_size

        indices = [
            word_to_index[token]
            for token in tokens
            if token in word_to_index
        ]

        num_full_blocks = len(indices) // sequence_length

        if num_full_blocks == 0:
            return np.empty(
                (0, sequence_length, vector_size),
                dtype=np.float32
            )

        usable_count = num_full_blocks * sequence_length
        indices = np.asarray(indices[:usable_count], dtype=np.int64)

        token_matrix = word_vectors.vectors[indices].astype(np.float32, copy=False)

        return token_matrix.reshape(
            num_full_blocks,
            sequence_length,
            vector_size
        )

    def collection_to_embedding_blocks(
        self,
        tokenized_texts: Sequence[Sequence[str]],
        embedding_model: Word2Vec,
        sequence_length: int,
    ) -> List[np.ndarray]:

        word_vectors = embedding_model.wv
        word_to_index = word_vectors.key_to_index
        vector_size = word_vectors.vector_size

        all_indices = []
        valid_lengths_per_text = []

        for token_list in tokenized_texts:
            start_length = len(all_indices)

            all_indices.extend(
                word_to_index[token]
                for token in token_list
                if token in word_to_index
            )

            valid_lengths_per_text.append(len(all_indices) - start_length)

        if len(all_indices) == 0:
            return [
                np.empty((0, sequence_length, vector_size), dtype=np.float32)
                for _ in tokenized_texts
            ]

        all_indices = np.asarray(all_indices, dtype=np.int64)

        all_vectors = word_vectors.vectors[all_indices].astype(np.float32, copy=False)

        blocks_per_text = []
        cursor = 0

        for valid_length in valid_lengths_per_text:
            num_full_blocks = valid_length // sequence_length
            usable_count = num_full_blocks * sequence_length

            if usable_count == 0:
                blocks = np.empty(
                    (0, sequence_length, vector_size),
                    dtype=np.float32
                )
            else:
                blocks = all_vectors[
                    cursor : cursor + usable_count
                ].reshape(
                    num_full_blocks,
                    sequence_length,
                    vector_size
                )

            blocks_per_text.append(blocks)
            cursor += valid_length

        return blocks_per_text

In [ ]:
"""Classifier model implementing a hybrid 1D-CNN, Bidirectional LSTM, and Attention architecture.

    Attributes:
        train: Builds, compiles, and trains the hybrid neural network with early stopping.
        config: ExperimentConfig instance containing layer dimensions, filter sizes, and learning rates.
        one_hot_labels: Custom 2-class translation logic derived from the 1 and 2 integer labels.
    """
class RCNNAClassifier:
    def train(self, features: np.ndarray, labels: np.ndarray, config: ExperimentConfig):
        model = Sequential()

        for i, kernel_size in enumerate(config.kernel_sizes):
            if i == 0:
                model.add(
                    Conv1D(
                        filters=config.num_filters,
                        kernel_size=kernel_size,
                        padding="valid",
                        activation="relu",
                        input_shape=(config.sequence_length, config.embedding_dimension),
                    )
                )
            else:
                model.add(
                    Conv1D(
                        filters=config.num_filters,
                        kernel_size=kernel_size,
                        padding="valid",
                        activation="relu",
                    )
                )
            model.add(MaxPooling1D(pool_size=config.pool_size))

        model.add(
            Bidirectional(
                LSTM(units=config.lstm_units, return_sequences=True),
                merge_mode="concat"
            )
        )

        model.add(
            Bidirectional(
                LSTM(units=config.lstm_units, return_sequences=False)
            )
        )

        model.add(Attention(return_sequences=True))
        model.add(Dropout(config.dropout_rate))
        model.add(Dense(config.num_classes, activation="sigmoid"))

        model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

        one_hot_labels = np.zeros((labels.shape[0], 2))
        one_hot_labels[:, 0] = 2 - labels
        one_hot_labels[:, 1] = labels - 1

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
            mode="min"
        )

        history = model.fit(
            features,
            one_hot_labels,
            validation_split=config.test_size,
            epochs=config.epochs,
            verbose=1,
            callbacks=[early_stop]
        )

        return model, history

"""Constructs balanced feature and label arrays by upsampling the minority class blocks.

    Attributes:
        imposter_a_blocks & imposter_b_blocks: Feature blocks representing individual authors.
        replicated_shorter_group: Resampled minority class lists to balance dataset ratios.
        features & labels: Shuffled arrays ready for neural network consumption.
    """
def build_balanced_training_pairs(
    imposter_a_blocks: List[np.ndarray],
    imposter_b_blocks: List[np.ndarray],
):
    imposter_groups = [imposter_a_blocks, imposter_b_blocks]
    lengths = [len(imposter_groups[0]), len(imposter_groups[1])]

    longer_index = lengths.index(max(lengths))
    shorter_index = lengths.index(min(lengths))

    replicated_shorter_group = []

    if lengths[shorter_index] > 0:
        for _ in range(lengths[longer_index] // lengths[shorter_index]):
            replicated_shorter_group += imposter_groups[shorter_index]

        remaining = lengths[longer_index] - len(replicated_shorter_group)
        if remaining > 0:
            replicated_shorter_group += random.sample(imposter_groups[shorter_index], remaining)

        imposter_groups[shorter_index] = replicated_shorter_group

    imposter_groups[0] = imposter_groups[0] + imposter_groups[0] + imposter_groups[0]
    imposter_groups[1] = imposter_groups[1] + imposter_groups[1] + imposter_groups[1]

    labels = np.asarray([1] * len(imposter_groups[0]) + [2] * len(imposter_groups[1]))
    features = np.asarray([item for group in imposter_groups for item in group])

    indices = np.arange(len(features))
    np.random.shuffle(indices)

    features = features[indices]
    labels = labels[indices]

    return features, labels, imposter_groups

In [ ]:
"""Analyzes text predictions to compute pairwise time-warp distances and detect anomalies.

    Attributes:
        build_distance_matrix: Batches neural network model predictions and runs FastDTW to build a symmetric distance matrix.
        summarize_and_detect_anomalies: Uses an Isolation Forest to flag structural text anomalies and computes distance percentiles.
        raw_prediction_map: Temporary store mapping file names to continuous neural prediction chunks.
    """
class DistanceAnalyzer:

    def build_distance_matrix(
        self,
        model: Sequential,
        blocks_by_text: List[np.ndarray],
        file_names: List[str],
        batch_factor: int,
    ):
        raw_prediction_map = {}

        all_blocks = []
        lengths = []
        valid_file_names = []

        for file_name, blocks in zip(file_names, blocks_by_text):
            blocks = list(blocks)

            if len(blocks) == 0:
                continue

            valid_file_names.append(file_name)
            lengths.append(len(blocks))
            all_blocks.extend(blocks)

        if len(all_blocks) == 0:
            return {}, np.zeros((0, 0))

        print(f"Running one batched prediction over {len(all_blocks)} Torah blocks...")

        all_predictions = model.predict(
            np.asarray(all_blocks),
            verbose=0
        )[:, 0]

        start = 0
        for file_name, length in zip(valid_file_names, lengths):
            end = start + length
            raw_prediction_map[file_name] = all_predictions[start:end]
            start = end

        grouped_prediction_map = {}

        for file_name, scores in raw_prediction_map.items():
            grouped_prediction_map[file_name] = [
                sum(group) / batch_factor
                for group in zip(*[iter(scores)] * batch_factor)
            ]

        valid_items = {
            name: values
            for name, values in grouped_prediction_map.items()
            if len(values) >= 2
        }

        valid_names = list(valid_items.keys())
        distance_matrix = np.zeros((len(valid_names), len(valid_names)))

        print(f"Calculating symmetric FastDTW matrix for {len(valid_names)} texts...")

        for i, name_a in enumerate(valid_names):
            for j in range(i + 1, len(valid_names)):
                name_b = valid_names[j]

                distance_value, _ = fastdtw(
                    valid_items[name_a],
                    valid_items[name_b]
                )

                distance_matrix[i, j] = distance_value
                distance_matrix[j, i] = distance_value

        return valid_items, distance_matrix

    def summarize_and_detect_anomalies(
        self,
        distance_matrix: np.ndarray,
        threshold: float,
        file_names: List[str],
        summary_percentile: int,
        lines_from_classified_text: List[str],
    ):
        isolation_forest = IsolationForest(n_estimators=1000, warm_start=True)
        isolation_forest.fit(distance_matrix)

        anomaly_predictions = isolation_forest.predict(distance_matrix)
        anomaly_scores = isolation_forest.decision_function(distance_matrix)

        flagged_indices = np.where(anomaly_scores < threshold)[0]
        flagged_names = [file_names[index] for index in flagged_indices]

        overlap_count = len(set(flagged_names) & set(lines_from_classified_text))

        row_sums = np.asarray([np.sum(row) for row in distance_matrix], dtype=float)

        if np.sum(row_sums) == 0:
            normalized_sums = row_sums
        else:
            normalized_sums = row_sums / np.sum(row_sums)

        _ = np.where(
            normalized_sums > np.percentile(normalized_sums, summary_percentile)
        )[0]

        return normalized_sums, anomaly_scores, anomaly_predictions, overlap_count

"""Handles persistence of experiment artifacts, saving metrics, reports, and training plots.

    Attributes:
        paths: PipelinePaths instance specifying the directory mapping for output storage.
        save_training_plots: Exports model accuracy and loss curve visualizations over training epochs.
        save_iteration_reports: Dumps current iteration state (dataframes, matrices, cluster labels) into CSV and NPY files.
    """
class ResultWriter:

    def __init__(self, paths: PipelinePaths):
        self.paths = paths

    def save_training_plots(self, history, iteration_number: int):
        plt.plot(history.history["accuracy"])
        plt.plot(history.history["val_accuracy"])
        plt.title("Model Accuracy")
        plt.ylabel("Accuracy")
        plt.xlabel("Epoch")
        plt.legend(["Train", "Validation"], loc="upper left")
        plt.savefig(self.paths.results_root / f"accuracy_{iteration_number}.png")
        plt.close()

        plt.plot(history.history["loss"])
        plt.plot(history.history["val_loss"])
        plt.title("Model Loss")
        plt.ylabel("Loss")
        plt.xlabel("Epoch")
        plt.legend(["Train", "Validation"], loc="upper left")
        plt.savefig(self.paths.results_root / f"loss_{iteration_number}.png")
        plt.close()

    def save_iteration_reports(
        self,
        state: PipelineState,
        config: ExperimentConfig,
    ):
        iteration_number = state.iteration_number
        lstm_units = config.lstm_units

        if state.forest_report:
            forest_df = pd.DataFrame(state.forest_report)
            forest_df.to_csv(
                self.paths.results_root / f"res_report_{iteration_number}_{lstm_units}.csv",
                index=False,
            )

        if state.summary_report:
            summary_df = pd.DataFrame(state.summary_report)
            summary_df.to_csv(
                self.paths.results_root / f"res_report_summ_{iteration_number}_{lstm_units}.csv",
                index=False,
            )

        if state.distance_label_report:
            dist_label_df = pd.DataFrame(state.distance_label_report)
            dist_label_df.to_csv(
                self.paths.results_root / f"res_report_Dist_{iteration_number}_{lstm_units}.csv",
                index=False,
            )

        if state.latest_distance_matrix is not None:
            np.save(
                self.paths.results_root / f"dist_mat_{iteration_number}.npy",
                state.latest_distance_matrix
            )

            dist_mat_df = pd.DataFrame(state.latest_distance_matrix)
            dist_mat_df.to_csv(
                self.paths.results_root / f"dist_mat_{iteration_number}.csv",
                index=False,
            )

In [ ]:
import json
import os
import traceback

"""Orchestrates the entire Deep Impostors pipeline execution, environment verification, and recovery logic.

    Attributes:
        paths & config: PipelinePaths and ExperimentConfig configurations managing directories and parameters.
        state & repository: Infrastructure components managing intermediate run data and corpus loading.
        validate_environment: Runs comprehensive preflight checks to ensure required Hebrew data structures and embeddings exist.
        run & run_single_pair: Execution entry points driving grid-search loops or target-pair analysis with automated memory optimization.
    """
class DeepImpostorsPipeline:
    def __init__(self, paths: PipelinePaths, config: ExperimentConfig):
        self.paths = paths
        self.config = config
        self.state = PipelineState()
        self.repository = CorpusRepository(paths, self.state)
        self.preprocessor = TextPreprocessor()
        self.embedding_trainer = EmbeddingTrainer(paths)
        self.sequence_embedder = SequenceEmbedder()
        self.classifier = RCNNAClassifier()
        self.distance_analyzer = DistanceAnalyzer()
        self.result_writer = ResultWriter(paths)

    def validate_environment(self, name_a: str | None = None, name_b: str | None = None):

        print("Running preflight checks before loading FastText...")

        required_dirs = {
            "project_root": self.paths.project_root,
            "torah_dir": self.paths.torah_dir,
            "imposters_dir": self.paths.imposters_dir,
            "imposters1_dir": self.paths.imposters1_dir,
            "results_root": self.paths.results_root,
        }

        for label, path in required_dirs.items():
            if not path.exists():
                raise FileNotFoundError(f"[Preflight] Missing required path: {label} -> {path}")
            if label != "results_root" and not path.is_dir():
                raise NotADirectoryError(f"[Preflight] Expected directory for {label}, got: {path}")

        if not self.paths.hebrew_fasttext_path.exists():
            raise FileNotFoundError(
                f"[Preflight] Missing Hebrew FastText vectors file: {self.paths.hebrew_fasttext_path}"
            )
        if not self.paths.hebrew_fasttext_path.is_file():
            raise FileNotFoundError(
                f"[Preflight] Hebrew FastText path is not a file: {self.paths.hebrew_fasttext_path}"
            )

        torah_files = sorted(self.paths.torah_dir.glob("*.txt"))
        if not torah_files:
            raise FileNotFoundError(
                f"[Preflight] No .txt files found in torah_dir: {self.paths.torah_dir}"
            )

        imp_a_dirs = sorted([p.name for p in self.paths.imposters_dir.iterdir() if p.is_dir()])
        imp_b_dirs = sorted([p.name for p in self.paths.imposters1_dir.iterdir() if p.is_dir()])

        if not imp_a_dirs:
            raise FileNotFoundError(
                f"[Preflight] No imposter author folders found in: {self.paths.imposters_dir}"
            )
        if not imp_b_dirs:
            raise FileNotFoundError(
                f"[Preflight] No imposter author folders found in: {self.paths.imposters1_dir}"
            )

        bad_imp_a = []
        for author in imp_a_dirs:
            txts = list((self.paths.imposters_dir / author).glob("*.txt"))
            if not txts:
                bad_imp_a.append(author)

        bad_imp_b = []
        for author in imp_b_dirs:
            txts = list((self.paths.imposters1_dir / author).glob("*.txt"))
            if not txts:
                bad_imp_b.append(author)

        if bad_imp_a:
            raise FileNotFoundError(
                f"[Preflight] These imposters folders in imposters have no .txt files: {bad_imp_a}"
            )
        if bad_imp_b:
            raise FileNotFoundError(
                f"[Preflight] These imposters folders in imposters1 have no .txt files: {bad_imp_b}"
            )

        if name_a is not None and name_a not in imp_a_dirs:
            raise ValueError(
                f"[Preflight] Imposter A '{name_a}' not found in imposters. Available: {imp_a_dirs}"
            )
        if name_b is not None and name_b not in imp_b_dirs:
            raise ValueError(
                f"[Preflight] Imposter B '{name_b}' not found in imposters1. Available: {imp_b_dirs}"
            )

        sample_torah = torah_files[0]
        if sample_torah.stat().st_size == 0:
            raise ValueError(f"[Preflight] Torah file is empty: {sample_torah}")

        if name_a is not None:
            sample_a = sorted((self.paths.imposters_dir / name_a).glob("*.txt"))[0]
            if sample_a.stat().st_size == 0:
                raise ValueError(f"[Preflight] Imposter A file is empty: {sample_a}")

        if name_b is not None:
            sample_b = sorted((self.paths.imposters1_dir / name_b).glob("*.txt"))[0]
            if sample_b.stat().st_size == 0:
                raise ValueError(f"[Preflight] Imposter B file is empty: {sample_b}")

        print("Preflight checks passed.")
        print(f"Torah books found: {len(torah_files)}")
        print(f"Imposters folders found: {len(imp_a_dirs)}")
        print(f"Imposters1 folders found: {len(imp_b_dirs)}")
        if name_a is not None:
            print(f"Chosen A exists: {name_a}")
        if name_b is not None:
            print(f"Chosen B exists: {name_b}")
        print(f"FastText file exists: {self.paths.hebrew_fasttext_path}")

    def load_progress(self) -> int:
        if not self.paths.progress_file.exists():
            return 1

        try:
            with open(self.paths.progress_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            last_completed = int(data.get("last_completed_iteration", 0))
            return last_completed + 1
        except Exception as e:
            print(f"Warning: could not read progress file. Starting from 1. Error: {e}")
            return 1

    def save_progress(self, completed_iteration: int, name_a: str, name_b: str):
        payload = {
            "last_completed_iteration": completed_iteration,
            "last_imposter_a": name_a,
            "last_imposter_b": name_b,
        }
        with open(self.paths.progress_file, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

    def reset_progress(self):
        if self.paths.progress_file.exists():
            self.paths.progress_file.unlink()
            print("Progress file deleted.")
        else:
            print("No progress file found.")

    def run(self, resume: bool = True):
        print("Starting full pipeline...")

        self.validate_environment()
        self.repository.load_torah_corpus()
        self.embedding_trainer._ensure_vectors_loaded()

        im_a = self.repository.list_imposter_names("imposters")
        im_b = self.repository.list_imposter_names("imposters1")
        all_pairs = [(a, b) for a in range(len(im_a)) for b in range(len(im_b))]
        max_iter = len(all_pairs)

        start_iteration = self.load_progress() if resume else 1
        self.state.iteration_number = start_iteration

        print(f"Total iterations planned: {max_iter}")
        print(f"Starting from iteration: {self.state.iteration_number}")

        while self.state.iteration_number <= max_iter:
            idx_a, idx_b = all_pairs[self.state.iteration_number - 1]
            name_a, name_b = im_a[idx_a], im_b[idx_b]

            print("\n" + "=" * 70)
            print(f"Iteration {self.state.iteration_number}: Torah vs {name_a} & {name_b}")
            print("=" * 70)

            try:
                self.repository.load_imposter_group_by_name(name_a, "imposters")
                self.repository.load_imposter_group_by_name(name_b, "imposters1")

                self._run_single_iteration(name_a, name_b)
                self.save_progress(self.state.iteration_number, name_a, name_b)
                self.state.iteration_number += 1

            except Exception as e:
                print("\n" + "!" * 70)
                print(f"Crash at iteration {self.state.iteration_number}")
                print(f"Pair: {name_a} vs {name_b}")
                print(f"Error: {e}")
                traceback.print_exc()
                print("Progress was saved up to the previous successful iteration.")
                print("Rerun the recovery cell to continue.")
                print("!" * 70)
                break

    def run_single_pair(self, name_a: str, name_b: str):
        print("Starting single-pair pipeline...")
        print(f"Chosen imposters: {name_a} vs {name_b}")

        self.validate_environment(name_a=name_a, name_b=name_b)
        self.repository.load_torah_corpus()
        self.embedding_trainer._ensure_vectors_loaded()

        self.repository.load_imposter_group_by_name(name_a, "imposters")
        self.repository.load_imposter_group_by_name(name_b, "imposters1")

        self.state.iteration_number = 1
        self._run_single_iteration(name_a, name_b)

        print("\nSingle-pair run finished successfully.")
        print("Saved files:")
        try:
            print(sorted(os.listdir(self.paths.results_root)))
        except Exception:
            pass

    def _run_single_iteration(self, name_a: str, name_b: str):
        im_a_texts = self.state.database[name_a]
        im_b_texts = self.state.database[name_b]
        torah_texts = list(self.state.database["torah"])
        torah_names = self.repository.list_torah_file_stems()

        print("\n[Stage] Preprocessing (Hebrew)")
        p_im_a = self.preprocessor.preprocess_collection(im_a_texts)
        p_im_b = self.preprocessor.preprocess_collection(im_b_texts)
        p_torah = self.preprocessor.preprocess_collection(torah_texts)

        print("Number of imposter A texts:", len(p_im_a))
        print("Number of imposter B texts:", len(p_im_b))
        print("Number of Torah books:", len(p_torah))
        print("Torah file names:", torah_names)

        print("\n[Stage] Hebrew embedding + fine-tuning")
        emb_model = self.embedding_trainer.train_word2vec(
            p_im_a,
            p_im_b,
            p_torah,
            self.config.embedding_dimension,
            self.config.embedding_algorithm,
        )

        print("\n[Stage] Building embedding blocks")
        t_blocks = self.sequence_embedder.collection_to_embedding_blocks(
            p_torah, emb_model, self.config.sequence_length
        )
        a_blocks = self.sequence_embedder.collection_to_embedding_blocks(
            p_im_a, emb_model, self.config.sequence_length
        )
        b_blocks = self.sequence_embedder.collection_to_embedding_blocks(
            p_im_b, emb_model, self.config.sequence_length
        )

        f_a = [b for blks in a_blocks for b in blks]
        f_b = [b for blks in b_blocks for b in blks]

        print("Flattened imposter A blocks:", len(f_a))
        print("Flattened imposter B blocks:", len(f_b))
        print("Number of Torah block groups:", len(t_blocks))

        train_X, train_Y, _ = build_balanced_training_pairs(f_a, f_b)

        print("train_X shape:", train_X.shape)
        print("train_Y shape:", train_Y.shape)

        print("\n[Stage] Training RCNNA")
        model, history = self.classifier.train(train_X, train_Y, self.config)
        self.result_writer.save_training_plots(history, self.state.iteration_number)

        print("\n[Stage] Calculating Distance Matrix (FastDTW)")
        preds, dist_mat = self.distance_analyzer.build_distance_matrix(
            model,
            t_blocks,
            torah_names,
            self.config.batch_factor,
        )
        valid_names = list(preds.keys())

        print("Valid names used in distance matrix:", valid_names)
        print("Distance matrix shape:", dist_mat.shape)

        summ, anom_s, anom_l, overlap = self.distance_analyzer.summarize_and_detect_anomalies(
            dist_mat,
            self.config.isolation_threshold,
            valid_names,
            self.config.summary_percentile,
            [],
        )

        print("\n[Stage] Isolation Forest results")
        print("Anomaly scores:", anom_s)
        print("Summary scores:", summ)
        print("Overlap count:", overlap)

        print("\n[Stage] K-Medoids")
        kmed = KMedoidsAdapter(n_clusters=2, random_state=0).fit(dist_mat)
        print("Cluster labels:", kmed.labels_)

        self.state.latest_distance_matrix = dist_mat
        self.state.latest_kmedoids_labels = kmed.labels_

        if "Names" not in self.state.forest_report:
            self.state.forest_report["Names"] = valid_names
            self.state.summary_report["Names"] = valid_names
            self.state.distance_label_report["Names"] = valid_names

        self.state.forest_report[self.state.iteration_number] = anom_s
        self.state.summary_report[self.state.iteration_number] = summ
        self.state.distance_label_report[self.state.iteration_number] = kmed.labels_

        self.result_writer.save_iteration_reports(self.state, self.config)

        # =========================
        # Safe memory cleanup
        # =========================
        import gc
        import tensorflow as tf

        del emb_model, model, history, preds, dist_mat, summ, anom_s, anom_l, kmed
        del p_im_a, p_im_b, p_torah
        del a_blocks, b_blocks, t_blocks
        del f_a, f_b, train_X, train_Y, valid_names

        gc.collect()
        tf.keras.backend.clear_session()

In [ ]:
"""Caches preprocessed Torah corpus tokens and file names in memory to bypass redundant tokenization.

    Attributes:
        _cached_torah_tokens: List storing cached, preprocessed Hebrew token sequences.
        _cached_torah_names: List tracking corresponding Torah chapter file stems.
    """
def _prepare_cached_torah(self):

    if hasattr(self, "_cached_torah_tokens") and self._cached_torah_tokens is not None:
        return self._cached_torah_tokens, self._cached_torah_names

    print("\n[Cache] Preprocessing Torah chapters once...")

    torah_texts = list(self.state.database["torah"])
    torah_names = self.repository.list_torah_file_stems()

    self._cached_torah_tokens = self.preprocessor.preprocess_collection(torah_texts)
    self._cached_torah_names = torah_names

    print(f"[Cache] Cached Torah texts: {len(self._cached_torah_tokens)}")

    return self._cached_torah_tokens, self._cached_torah_names

"""Executes a single pipeline step utilizing cached Torah vectors to accelerate loop iterations.

    Attributes:
        p_torah & torah_names: Fetched seamlessly from cache rather than parsed statically each run.
        f_a & f_b: Flattened 3D token block groupings used to train and compile downstream classifiers.
        gc.collect: Explicit memory sweeps preventing GPU/RAM memory leaks between iteration sequences.
    """
def _optimized_run_single_iteration(self, name_a: str, name_b: str):
    im_a_texts = self.state.database[name_a]
    im_b_texts = self.state.database[name_b]

    print("\n[Stage] Preprocessing (Hebrew)")
    p_im_a = self.preprocessor.preprocess_collection(im_a_texts)
    p_im_b = self.preprocessor.preprocess_collection(im_b_texts)

    p_torah, torah_names = self._prepare_cached_torah()

    print("Number of imposter A texts:", len(p_im_a))
    print("Number of imposter B texts:", len(p_im_b))
    print("Number of Torah chapter files:", len(p_torah))
    print("Torah file names:", torah_names)

    print("\n[Stage] Hebrew embedding + fine-tuning")
    emb_model = self.embedding_trainer.train_word2vec(
        p_im_a,
        p_im_b,
        p_torah,
        self.config.embedding_dimension,
        self.config.embedding_algorithm,
    )

    print("\n[Stage] Building embedding blocks")
    t_blocks = self.sequence_embedder.collection_to_embedding_blocks(
        p_torah, emb_model, self.config.sequence_length
    )
    a_blocks = self.sequence_embedder.collection_to_embedding_blocks(
        p_im_a, emb_model, self.config.sequence_length
    )
    b_blocks = self.sequence_embedder.collection_to_embedding_blocks(
        p_im_b, emb_model, self.config.sequence_length
    )

    f_a = [b for blks in a_blocks for b in blks]
    f_b = [b for blks in b_blocks for b in blks]

    print("Flattened imposter A blocks:", len(f_a))
    print("Flattened imposter B blocks:", len(f_b))
    print("Number of Torah block groups:", len(t_blocks))

    train_X, train_Y, _ = build_balanced_training_pairs(f_a, f_b)

    print("train_X shape:", train_X.shape)
    print("train_Y shape:", train_Y.shape)

    print("\n[Stage] Training RCNNA")
    model, history = self.classifier.train(train_X, train_Y, self.config)
    self.result_writer.save_training_plots(history, self.state.iteration_number)

    print("\n[Stage] Calculating Distance Matrix (FastDTW)")
    preds, dist_mat = self.distance_analyzer.build_distance_matrix(
        model,
        t_blocks,
        torah_names,
        self.config.batch_factor,
    )

    valid_names = list(preds.keys())

    print("Valid names used in distance matrix:", valid_names)
    print("Distance matrix shape:", dist_mat.shape)

    summ, anom_s, anom_l, overlap = self.distance_analyzer.summarize_and_detect_anomalies(
        dist_mat,
        self.config.isolation_threshold,
        valid_names,
        self.config.summary_percentile,
        [],
    )

    print("\n[Stage] Isolation Forest results")
    print("Anomaly scores:", anom_s)
    print("Summary scores:", summ)
    print("Overlap count:", overlap)

    print("\n[Stage] K-Medoids")
    kmed = KMedoidsAdapter(n_clusters=2, random_state=0).fit(dist_mat)
    print("Cluster labels:", kmed.labels_)

    self.state.latest_distance_matrix = dist_mat
    self.state.latest_kmedoids_labels = kmed.labels_

    if "Names" not in self.state.forest_report:
        self.state.forest_report["Names"] = valid_names
        self.state.summary_report["Names"] = valid_names
        self.state.distance_label_report["Names"] = valid_names

    self.state.forest_report[self.state.iteration_number] = anom_s
    self.state.summary_report[self.state.iteration_number] = summ
    self.state.distance_label_report[self.state.iteration_number] = kmed.labels_

    self.result_writer.save_iteration_reports(self.state, self.config)

    import gc
    import tensorflow as tf

    del emb_model, model, history, preds, dist_mat, summ, anom_s, anom_l, kmed
    del p_im_a, p_im_b
    del a_blocks, b_blocks, t_blocks
    del f_a, f_b, train_X, train_Y, valid_names

    gc.collect()
    tf.keras.backend.clear_session()


DeepImpostorsPipeline._prepare_cached_torah = _prepare_cached_torah
DeepImpostorsPipeline._run_single_iteration = _optimized_run_single_iteration


------------------------
------------------------
------------------------


===== Full Execution Cell ====


------------------------
------------------------
------------------------

**Note**: If the program crashed and you need to resume from the last checkpoint, run the cell below instead.

In [ ]:
"""
Main Execution Script for the Deep Impostors Pipeline.

Initializes the project directories, establishes hyperparameter overrides,
resets any lingering iteration logs, and triggers a clean, structured
evaluation loop across all author-imposter text pairings.
"""
from pathlib import Path

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_results/"),
)

config = ExperimentConfig(
    embedding_dimension=300,
    chunk_size=100,
    embedding_algorithm="hebrew_fasttext_vec",
)

pipeline = DeepImpostorsPipeline(paths, config)

pipeline.reset_progress()

pipeline.run(resume=False)


------------------------
------------------------
------------------------


===== **Crash Recovery Cell** ====


------------------------
------------------------
------------------------




In a new session after a crash:

1. Mount Google Drive.

2. Run all preceding cells, excluding the Full Execution cell.

3. Run this Crash Recovery cell.

In [ ]:
"""
Pipeline Recovery and Execution Script.

Initializes the environment using existing configurations and triggers
the pipeline while reading the progress file to safely resume execution
from the last successful author-imposter iteration.
"""
from pathlib import Path

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_results/"),
)

config = ExperimentConfig(
    embedding_dimension=300,
    chunk_size=100,
    embedding_algorithm="hebrew_fasttext_vec",
)

pipeline = DeepImpostorsPipeline(paths, config)

pipeline.run(resume=True)

## Recovery Sanity Check — How to Run and What It Tests

This section tests the pipeline recovery mechanism.

The goal is to verify that the pipeline can recover correctly after an unexpected crash or interruption.

The pipeline does **not** save the iteration that crashed.  
Instead, it saves only the **last iteration that finished successfully** into:

```python
progress.json

In [ ]:
import json
import os
import traceback
"""
Sanity-check extension of the DeepImpostorsPipeline.

This version is intended only for recovery validation and debugging.
It adds helper methods that simulate an intentional crash and verify
that the pipeline correctly resumes from the saved progress state.

The production pipeline uses the standard `run()` and `resume=True`
workflow, while the additional methods in this class are provided
exclusively for controlled sanity tests.

These tests verify that:
    - Progress is saved only after successful iterations.
    - Failed iterations are not marked as completed.
    - The pipeline resumes from the correct iteration after a crash.

This class should not be used as part of the regular experiment.
"""
class DeepImpostorsPipeline:
    def __init__(self, paths: PipelinePaths, config: ExperimentConfig):
        self.paths = paths
        self.config = config
        self.state = PipelineState()
        self.repository = CorpusRepository(paths, self.state)
        self.preprocessor = TextPreprocessor()
        self.embedding_trainer = EmbeddingTrainer(paths)
        self.sequence_embedder = SequenceEmbedder()
        self.classifier = RCNNAClassifier()
        self.distance_analyzer = DistanceAnalyzer()
        self.result_writer = ResultWriter(paths)

    def validate_environment(self, name_a: str | None = None, name_b: str | None = None):
        print("Running preflight checks before loading FastText...")

        required_dirs = {
            "project_root": self.paths.project_root,
            "torah_dir": self.paths.torah_dir,
            "imposters_dir": self.paths.imposters_dir,
            "imposters1_dir": self.paths.imposters1_dir,
            "results_root": self.paths.results_root,
        }

        for label, path in required_dirs.items():
            if not path.exists():
                raise FileNotFoundError(f"[Preflight] Missing required path: {label} -> {path}")
            if label != "results_root" and not path.is_dir():
                raise NotADirectoryError(f"[Preflight] Expected directory for {label}, got: {path}")

        if not self.paths.hebrew_fasttext_path.exists():
            raise FileNotFoundError(
                f"[Preflight] Missing Hebrew FastText vectors file: {self.paths.hebrew_fasttext_path}"
            )
        if not self.paths.hebrew_fasttext_path.is_file():
            raise FileNotFoundError(
                f"[Preflight] Hebrew FastText path is not a file: {self.paths.hebrew_fasttext_path}"
            )

        torah_files = sorted(self.paths.torah_dir.glob("*.txt"))
        if not torah_files:
            raise FileNotFoundError(
                f"[Preflight] No .txt files found in torah_dir: {self.paths.torah_dir}"
            )

        imp_a_dirs = sorted([p.name for p in self.paths.imposters_dir.iterdir() if p.is_dir()])
        imp_b_dirs = sorted([p.name for p in self.paths.imposters1_dir.iterdir() if p.is_dir()])

        if not imp_a_dirs:
            raise FileNotFoundError(
                f"[Preflight] No imposter author folders found in: {self.paths.imposters_dir}"
            )
        if not imp_b_dirs:
            raise FileNotFoundError(
                f"[Preflight] No imposter author folders found in: {self.paths.imposters1_dir}"
            )

        bad_imp_a = []
        for author in imp_a_dirs:
            txts = list((self.paths.imposters_dir / author).glob("*.txt"))
            if not txts:
                bad_imp_a.append(author)

        bad_imp_b = []
        for author in imp_b_dirs:
            txts = list((self.paths.imposters1_dir / author).glob("*.txt"))
            if not txts:
                bad_imp_b.append(author)

        if bad_imp_a:
            raise FileNotFoundError(
                f"[Preflight] These imposters folders in imposters have no .txt files: {bad_imp_a}"
            )
        if bad_imp_b:
            raise FileNotFoundError(
                f"[Preflight] These imposters folders in imposters1 have no .txt files: {bad_imp_b}"
            )

        if name_a is not None and name_a not in imp_a_dirs:
            raise ValueError(
                f"[Preflight] Imposter A '{name_a}' not found in imposters. Available: {imp_a_dirs}"
            )
        if name_b is not None and name_b not in imp_b_dirs:
            raise ValueError(
                f"[Preflight] Imposter B '{name_b}' not found in imposters1. Available: {imp_b_dirs}"
            )

        sample_torah = torah_files[0]
        if sample_torah.stat().st_size == 0:
            raise ValueError(f"[Preflight] Torah file is empty: {sample_torah}")

        if name_a is not None:
            sample_a = sorted((self.paths.imposters_dir / name_a).glob("*.txt"))[0]
            if sample_a.stat().st_size == 0:
                raise ValueError(f"[Preflight] Imposter A file is empty: {sample_a}")

        if name_b is not None:
            sample_b = sorted((self.paths.imposters1_dir / name_b).glob("*.txt"))[0]
            if sample_b.stat().st_size == 0:
                raise ValueError(f"[Preflight] Imposter B file is empty: {sample_b}")

        print("Preflight checks passed.")
        print(f"Torah books found: {len(torah_files)}")
        print(f"Imposters folders found: {len(imp_a_dirs)}")
        print(f"Imposters1 folders found: {len(imp_b_dirs)}")
        if name_a is not None:
            print(f"Chosen A exists: {name_a}")
        if name_b is not None:
            print(f"Chosen B exists: {name_b}")
        print(f"FastText file exists: {self.paths.hebrew_fasttext_path}")

    def load_progress(self) -> int:
        if not self.paths.progress_file.exists():
            return 1

        try:
            with open(self.paths.progress_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            last_completed = int(data.get("last_completed_iteration", 0))
            return last_completed + 1
        except Exception as e:
            print(f"Warning: could not read progress file. Starting from 1. Error: {e}")
            return 1

    def save_progress(self, completed_iteration: int, name_a: str, name_b: str):
        payload = {
            "last_completed_iteration": completed_iteration,
            "last_imposter_a": name_a,
            "last_imposter_b": name_b,
        }
        with open(self.paths.progress_file, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

    def reset_progress(self):
        """
        מוחק את קובץ ה-progress כדי להתחיל מחדש.
        """
        if self.paths.progress_file.exists():
            self.paths.progress_file.unlink()
            print("Progress file deleted.")
        else:
            print("No progress file found.")

    def run(self, resume: bool = True):
        print("Starting full pipeline...")

        self.validate_environment()
        self.repository.load_torah_corpus()
        self.embedding_trainer._ensure_vectors_loaded()

        im_a = self.repository.list_imposter_names("imposters")
        im_b = self.repository.list_imposter_names("imposters1")
        all_pairs = [(a, b) for a in range(len(im_a)) for b in range(len(im_b))]
        max_iter = len(all_pairs)

        start_iteration = self.load_progress() if resume else 1
        self.state.iteration_number = start_iteration

        print(f"Total iterations planned: {max_iter}")
        print(f"Starting from iteration: {self.state.iteration_number}")

        while self.state.iteration_number <= max_iter:
            idx_a, idx_b = all_pairs[self.state.iteration_number - 1]
            name_a, name_b = im_a[idx_a], im_b[idx_b]

            print("\n" + "=" * 70)
            print(f"Iteration {self.state.iteration_number}: Torah vs {name_a} & {name_b}")
            print("=" * 70)

            try:
                self.repository.load_imposter_group_by_name(name_a, "imposters")
                self.repository.load_imposter_group_by_name(name_b, "imposters1")

                self._run_single_iteration(name_a, name_b)

                self.save_progress(self.state.iteration_number, name_a, name_b)
                self.state.iteration_number += 1

            except Exception as e:
                print("\n" + "!" * 70)
                print(f"Crash at iteration {self.state.iteration_number}")
                print(f"Pair: {name_a} vs {name_b}")
                print(f"Error: {e}")
                traceback.print_exc()
                print("Progress was saved up to the previous successful iteration.")
                print("You can rerun pipeline.run(resume=True) to continue.")
                print("!" * 70)
                break

    def run_single_pair(self, name_a: str, name_b: str):
        print("Starting single-pair pipeline...")
        print(f"Chosen imposters: {name_a} vs {name_b}")

        self.validate_environment(name_a=name_a, name_b=name_b)
        self.repository.load_torah_corpus()
        self.embedding_trainer._ensure_vectors_loaded()

        self.repository.load_imposter_group_by_name(name_a, "imposters")
        self.repository.load_imposter_group_by_name(name_b, "imposters1")

        self.state.iteration_number = 1
        self._run_single_iteration(name_a, name_b)

        print("\nSingle-pair run finished successfully.")
        print("Saved files:")
        try:
            print(sorted(os.listdir(self.paths.results_root)))
        except Exception:
            pass

    def run_recovery_sanity_check(self, max_iterations: int = 5, crash_at_iteration: int = 3):
        print("Starting recovery sanity check...")

        self.validate_environment()
        self.repository.load_torah_corpus()
        self.embedding_trainer._ensure_vectors_loaded()

        im_a = self.repository.list_imposter_names("imposters")
        im_b = self.repository.list_imposter_names("imposters1")
        all_pairs = [(a, b) for a in range(len(im_a)) for b in range(len(im_b))]
        max_iter = min(len(all_pairs) // 2, max_iterations)

        self.state.iteration_number = 1

        print(f"Sanity check will run up to iteration {max_iter}")
        print(f"Simulated crash at iteration {crash_at_iteration}")

        while self.state.iteration_number <= max_iter:
            idx_a, idx_b = all_pairs[self.state.iteration_number - 1]
            name_a, name_b = im_a[idx_a], im_b[idx_b]

            print("\n" + "=" * 70)
            print(f"Iteration {self.state.iteration_number}: Torah vs {name_a} & {name_b}")
            print("=" * 70)

            try:
                if self.state.iteration_number == crash_at_iteration:
                    raise RuntimeError(
                        f"Simulated crash at iteration {self.state.iteration_number}"
                    )

                self.repository.load_imposter_group_by_name(name_a, "imposters")
                self.repository.load_imposter_group_by_name(name_b, "imposters1")

                self._run_single_iteration(name_a, name_b)

                self.save_progress(self.state.iteration_number, name_a, name_b)
                self.state.iteration_number += 1

            except Exception as e:
                print("\n" + "!" * 70)
                print(f"Crash at iteration {self.state.iteration_number}")
                print(f"Pair: {name_a} vs {name_b}")
                print(f"Error: {e}")
                traceback.print_exc()
                print("Progress was saved up to the previous successful iteration.")
                print("!" * 70)
                break

    def resume_from_progress_for_sanity(self, max_iterations: int = 5):
        print("Resuming recovery sanity check...")

        self.validate_environment()
        self.repository.load_torah_corpus()
        self.embedding_trainer._ensure_vectors_loaded()

        im_a = self.repository.list_imposter_names("imposters")
        im_b = self.repository.list_imposter_names("imposters1")
        all_pairs = [(a, b) for a in range(len(im_a)) for b in range(len(im_b))]
        max_iter = min(len(all_pairs) // 2, max_iterations)

        self.state.iteration_number = self.load_progress()
        print(f"Resuming from iteration {self.state.iteration_number}")

        while self.state.iteration_number <= max_iter:
            idx_a, idx_b = all_pairs[self.state.iteration_number - 1]
            name_a, name_b = im_a[idx_a], im_b[idx_b]

            print("\n" + "=" * 70)
            print(f"Iteration {self.state.iteration_number}: Torah vs {name_a} & {name_b}")
            print("=" * 70)

            try:
                self.repository.load_imposter_group_by_name(name_a, "imposters")
                self.repository.load_imposter_group_by_name(name_b, "imposters1")

                self._run_single_iteration(name_a, name_b)

                self.save_progress(self.state.iteration_number, name_a, name_b)
                self.state.iteration_number += 1

            except Exception as e:
                print("\n" + "!" * 70)
                print(f"Crash at iteration {self.state.iteration_number}")
                print(f"Pair: {name_a} vs {name_b}")
                print(f"Error: {e}")
                traceback.print_exc()
                print("Progress was saved up to the previous successful iteration.")
                print("!" * 70)
                break

    def _run_single_iteration(self, name_a: str, name_b: str):
        im_a_texts = self.state.database[name_a]
        im_b_texts = self.state.database[name_b]
        torah_texts = list(self.state.database["torah"])
        torah_names = self.repository.list_torah_file_stems()

        print("\n[Stage] Preprocessing (Hebrew)")
        p_im_a = self.preprocessor.preprocess_collection(im_a_texts)
        p_im_b = self.preprocessor.preprocess_collection(im_b_texts)
        p_torah = self.preprocessor.preprocess_collection(torah_texts)

        print("Number of imposter A texts:", len(p_im_a))
        print("Number of imposter B texts:", len(p_im_b))
        print("Number of Torah books:", len(p_torah))
        print("Torah file names:", torah_names)

        print("\n[Stage] Hebrew embedding + fine-tuning")
        emb_model = self.embedding_trainer.train_word2vec(
            p_im_a,
            p_im_b,
            p_torah,
            self.config.embedding_dimension,
            self.config.embedding_algorithm,
        )

        print("\n[Stage] Building embedding blocks")
        t_blocks = self.sequence_embedder.collection_to_embedding_blocks(
            p_torah, emb_model, self.config.sequence_length
        )
        a_blocks = self.sequence_embedder.collection_to_embedding_blocks(
            p_im_a, emb_model, self.config.sequence_length
        )
        b_blocks = self.sequence_embedder.collection_to_embedding_blocks(
            p_im_b, emb_model, self.config.sequence_length
        )

        f_a = [b for blks in a_blocks for b in blks]
        f_b = [b for blks in b_blocks for b in blks]

        print("Flattened imposter A blocks:", len(f_a))
        print("Flattened imposter B blocks:", len(f_b))
        print("Number of Torah block groups:", len(t_blocks))

        train_X, train_Y, _ = build_balanced_training_pairs(f_a, f_b)

        print("train_X shape:", train_X.shape)
        print("train_Y shape:", train_Y.shape)

        print("\n[Stage] Training RCNNA")
        model, history = self.classifier.train(train_X, train_Y, self.config)
        self.result_writer.save_training_plots(history, self.state.iteration_number)

        print("\n[Stage] Calculating Distance Matrix (FastDTW)")
        preds, dist_mat = self.distance_analyzer.build_distance_matrix(
            model,
            t_blocks,
            torah_names,
            self.config.batch_factor,
        )
        valid_names = list(preds.keys())

        print("Valid names used in distance matrix:", valid_names)
        print("Distance matrix shape:", dist_mat.shape)

        summ, anom_s, anom_l, overlap = self.distance_analyzer.summarize_and_detect_anomalies(
            dist_mat,
            self.config.isolation_threshold,
            valid_names,
            self.config.summary_percentile,
            [],
        )

        print("\n[Stage] Isolation Forest results")
        print("Anomaly scores:", anom_s)
        print("Summary scores:", summ)
        print("Overlap count:", overlap)

        print("\n[Stage] K-Medoids")
        kmed = KMedoidsAdapter(n_clusters=2, random_state=0).fit(dist_mat)
        print("Cluster labels:", kmed.labels_)

        self.state.latest_distance_matrix = dist_mat
        self.state.latest_kmedoids_labels = kmed.labels_

        if "Names" not in self.state.forest_report:
            self.state.forest_report["Names"] = valid_names
            self.state.summary_report["Names"] = valid_names
            self.state.distance_label_report["Names"] = valid_names

        self.state.forest_report[self.state.iteration_number] = anom_s
        self.state.summary_report[self.state.iteration_number] = summ
        self.state.distance_label_report[self.state.iteration_number] = kmed.labels_

        self.result_writer.save_iteration_reports(self.state, self.config)

        # =========================
        # Safe memory cleanup
        # =========================
        import gc
        import tensorflow as tf

        del emb_model, model, history, preds, dist_mat, summ, anom_s, anom_l, kmed
        del p_im_a, p_im_b, p_torah
        del a_blocks, b_blocks, t_blocks
        del f_a, f_b, train_X, train_Y, valid_names

        gc.collect()
        tf.keras.backend.clear_session()

In [ ]:

"""
Pipeline Recovery Sanity Check - Phase 1 (Simulation).

Initializes a targeted workspace to simulate a controlled execution crash
at a specific iteration, validating that the pipeline safely catches the error
and writes the exact step-state to the tracking JSON file.
"""
import json
from pathlib import Path

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_recovery_sanity_check/"),
)

config = ExperimentConfig(
    embedding_dimension=300,
    chunk_size=100,
    embedding_algorithm="hebrew_fasttext_vec",
)

pipeline = DeepImpostorsPipeline(paths, config)

print("=" * 80)
print("RECOVERY SANITY CHECK - STEP 1")
print("We intentionally crash at iteration 3")
print("=" * 80)

pipeline.reset_progress()

pipeline.run_recovery_sanity_check(
    max_iterations=5,
    crash_at_iteration=3,
)

print("\n" + "=" * 80)
print("CHECKING progress.json AFTER CRASH")
print("=" * 80)

if paths.progress_file.exists():
    with open(paths.progress_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(data)
else:
    print("No progress file found.")

sanity check המשך מהריצה האחרונה

In [ ]:
"""
Pipeline Recovery Sanity Check - Phase 2 (Validation).

Resumes the execution loop from the exact step logged during the simulated
crash, verifying that the pipeline correctly parses the tracking JSON, recovers
its internal state, and runs through to completion.
"""

from pathlib import Path

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_recovery_sanity_check/"),
)

config = ExperimentConfig(
    embedding_dimension=300,
    chunk_size=100,
    embedding_algorithm="hebrew_fasttext_vec",
)

pipeline = DeepImpostorsPipeline(paths, config)

print("=" * 80)
print("RECOVERY SANITY CHECK - STEP 2")
print("Now resume after simulated crash")
print("=" * 80)

pipeline.resume_from_progress_for_sanity(max_iterations=5)

sanity check single pair imposter check

In [ ]:
"""
Single-Pair Pipeline Execution and Sanity Check.

Runs a focused, isolated iteration comparing a specific author pairing
(e.g., Alharizi vs. Berechiah) against the Torah corpus, verifying the
end-to-end extraction, embedding, classification, and file generation pipeline.
"""

IMPOSTER_A_NAME = "יהודה אלחריזי"
IMPOSTER_B_NAME = "ברכיה הנקדן"


paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/hebrew_single_pair_sanity_check/"),
)


config = ExperimentConfig(
    embedding_dimension=300,
    chunk_size=100,
    embedding_algorithm="hebrew_fasttext_vec",
)

pipeline = DeepImpostorsPipeline(paths, config)

print("=" * 80)
print("SANITY CHECK RUN")
print(f"Chosen pair: {IMPOSTER_A_NAME}  vs  {IMPOSTER_B_NAME}")
print(f"Project root: {paths.project_root}")
print(f"Results root: {paths.results_root}")
print("=" * 80)

pipeline.run_single_pair(IMPOSTER_A_NAME, IMPOSTER_B_NAME)

print("\n" + "=" * 80)
print("SANITY CHECK FINISHED")
print("Generated files:")
try:
    for f in sorted(os.listdir(paths.results_root)):
        print("-", f)
except Exception as e:
    print("Could not list files:", e)
print("=" * 80)

# ============================================================
# DEBUG / SANITY CHECK: Validate Training Labels and Keras Split
# ============================================================

In [ ]:
"""
Sanity check for the training data before model training.

This cell reproduces the preprocessing and dataset construction
steps used by the main pipeline for a single pair of imposters.

The purpose is to verify that:

    - The selected corpora are loaded correctly.
    - Preprocessing produces reasonable token counts.
    - Embedding blocks are generated as expected.
    - The balanced training dataset has the expected size.
    - Class labels are properly balanced.
    - Keras validation_split does not accidentally receive
      a highly skewed label distribution.
    - One-hot encoding matches the format expected by the classifier.

This is a diagnostic/debugging cell only.

It does not perform a full experiment and does not generate
the final authorship analysis results.

Use this cell whenever training behaves unexpectedly
(e.g. val_accuracy remains near zero or the model fails to learn). """

from pathlib import Path
import numpy as np
from collections import Counter

IMPOSTER_A_NAME = "יהודה אלחריזי"
IMPOSTER_B_NAME = "ברכיה הנקדן"

paths = PipelinePaths(
    project_root=Path("/content/drive/MyDrive/deep_imposters_final_project/"),
    results_root=Path("/content/drive/MyDrive/deep_imposters_final_project/debug_validation_check/"),
)

config = ExperimentConfig(
    embedding_dimension=300,
    chunk_size=100,
    embedding_algorithm="hebrew_fasttext_vec",
)

pipeline = DeepImpostorsPipeline(paths, config)
pipeline.validate_environment(name_a=IMPOSTER_A_NAME, name_b=IMPOSTER_B_NAME)
pipeline.repository.load_torah_corpus()
pipeline.repository.load_imposter_group_by_name(IMPOSTER_A_NAME, "imposters")
pipeline.repository.load_imposter_group_by_name(IMPOSTER_B_NAME, "imposters1")
im_a_texts = pipeline.state.database[IMPOSTER_A_NAME]
im_b_texts = pipeline.state.database[IMPOSTER_B_NAME]
torah_texts = list(pipeline.state.database["torah"])
torah_names = pipeline.repository.list_torah_file_stems()

print("=" * 80)
print("BASIC COUNTS")
print("=" * 80)
print("Imposter A name:", IMPOSTER_A_NAME)
print("Imposter B name:", IMPOSTER_B_NAME)
print("Imposter A text files:", len(im_a_texts))
print("Imposter B text files:", len(im_b_texts))
print("Torah books:", len(torah_texts))
print("Torah names:", torah_names)

p_im_a = pipeline.preprocessor.preprocess_collection(im_a_texts)
p_im_b = pipeline.preprocessor.preprocess_collection(im_b_texts)
p_torah = pipeline.preprocessor.preprocess_collection(torah_texts)

print("\n" + "=" * 80)
print("TOKEN COUNTS AFTER PREPROCESSING")
print("=" * 80)
print("Imposter A token counts per file:", [len(x) for x in p_im_a])
print("Imposter B token counts per file:", [len(x) for x in p_im_b])
print("Torah token counts per book:", dict(zip(torah_names, [len(x) for x in p_torah])))

# embedding
emb_model = pipeline.embedding_trainer.train_word2vec(
    p_im_a,
    p_im_b,
    p_torah,
    config.embedding_dimension,
    config.embedding_algorithm,
)

# blocks
a_blocks = pipeline.sequence_embedder.collection_to_embedding_blocks(
    p_im_a, emb_model, config.sequence_length
)
b_blocks = pipeline.sequence_embedder.collection_to_embedding_blocks(
    p_im_b, emb_model, config.sequence_length
)
t_blocks = pipeline.sequence_embedder.collection_to_embedding_blocks(
    p_torah, emb_model, config.sequence_length
)

print("\n" + "=" * 80)
print("BLOCK COUNTS")
print("=" * 80)
print("Imposter A block counts per file:", [len(x) for x in a_blocks])
print("Imposter B block counts per file:", [len(x) for x in b_blocks])
print("Torah block counts per book:", dict(zip(torah_names, [len(x) for x in t_blocks])))

f_a = [b for blks in a_blocks for b in blks]
f_b = [b for blks in b_blocks for b in blks]

print("Flattened imposter A blocks:", len(f_a))
print("Flattened imposter B blocks:", len(f_b))

train_X, train_Y, _ = build_balanced_training_pairs(f_a, f_b)

print("\n" + "=" * 80)
print("TRAIN DATA SHAPES")
print("=" * 80)
print("train_X shape:", train_X.shape)
print("train_Y shape:", train_Y.shape)

label_counts_total = Counter(train_Y.tolist())
print("\nTotal label distribution:", dict(label_counts_total))
num_samples = len(train_Y)
val_size = int(num_samples * config.test_size)
train_size = num_samples - val_size

y_train_part = train_Y[:train_size]
y_val_part = train_Y[train_size:]

print("\n" + "=" * 80)
print("SIMULATED KERAS VALIDATION SPLIT")
print("=" * 80)
print("config.test_size:", config.test_size)
print("Total samples:", num_samples)
print("Train part size:", len(y_train_part))
print("Validation part size:", len(y_val_part))

print("Train label distribution:", dict(Counter(y_train_part.tolist())))
print("Validation label distribution:", dict(Counter(y_val_part.tolist())))

print("\nFirst 40 labels in full train_Y:")
print(train_Y[:40])

print("\nLast 40 labels in full train_Y:")
print(train_Y[-40:])
one_hot_labels = np.zeros((train_Y.shape[0], 2))
one_hot_labels[:, 0] = 2 - train_Y
one_hot_labels[:, 1] = train_Y - 1

print("\n" + "=" * 80)
print("ONE-HOT LABEL EXAMPLES")
print("=" * 80)
print("First 10 raw labels:", train_Y[:10])
print("First 10 one-hot rows:")
print(one_hot_labels[:10])

print("\nLast 10 raw labels:", train_Y[-10:])
print("Last 10 one-hot rows:")
print(one_hot_labels[-10:])

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("If validation distribution is very skewed or contains almost one class only,")
print("that can explain val_accuracy = 0.")
print("If labels are already mixed well, then the issue is probably small data / weak generalization.")